<div dir="rtl">
<h1>شمارهٔ سطر یاد نمی‌گیرد؛ عددهای سطر یاد می‌گیرند</h1>
<p>درس 29 از 76 · یک شناسه چطور به بردار قابل آموزش تبدیل می‌شود؟ · <code dir="ltr">25-embedding</code></p>
<p><a target="_self" href="http://127.0.0.1:8000/part-04/chapter-03/25-embedding.html">📖 بازگشت به همین درس</a></p>
<p>Lookup را خودتان بنویسید و Gradient سطرهای تکراری را در مدل واقعی ببینید.</p><p>پیش‌نیاز: 17-autograd،18-module و21-tokenizer؛ Embedding در درس جاری.</p>
<p>این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو Cell با برچسب TODO را خودتان کامل کنید. پیام INCOMPLETE یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p>از بالا به پایین اجرا کنید. پس از تغییر هر تابع، Cell آن و سپس Cell آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code>Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl">
<h2>قبل از اجرا، پیش‌بینی کنید</h2>
<p>برای IDهای [1,3,1] و Loss برابر مجموع خروجی، کدام سطر جدول دو سهم Gradient می‌گیرد و کدام سطرها هیچ سهمی ندارند؟</p>
</div>

<div dir="rtl"><p>پیش‌بینی من: …</p></div>

In [ ]:
import torch
from torch import nn
from mini_gpt.stages.v1 import TokenOnly
torch.manual_seed(7)
table = torch.arange(15, dtype=torch.float32).reshape(5, 3)
ids = torch.tensor([[1, 3, 1]], dtype=torch.long)
print("Known table:\n", table, "IDs:", ids)

<div dir="rtl">
<h2>این بار شما کد بنویسید</h2>
<p>تابع lookup(table,ids) با Indexing سطرهای جدول را انتخاب کند. table شکل (V,C) و ids شکل (B,T) دارد. خروجی باید (B,T,C) باشد؛ IDها معتبر و long هستند.</p>
</div>

In [ ]:
def lookup(table, ids):
    # TODO: preserve both ID axes and add the feature axis
    return None

In [ ]:
def test_exercise():
    result = lookup(table, ids)
    if result is None:
        return False
    assert tuple(result.shape) == (1, 3, 3)
    assert result.tolist() == [[[3., 4., 5.], [9., 10., 11.], [3., 4., 5.]]]
    assert torch.equal(result[0, 0], result[0, 2])
    assert tuple(lookup(table, torch.tensor([[0], [4]])).shape) == (2, 1, 3)
    return True

exercise_complete = test_exercise()
print('PASS' if exercise_complete else 'INCOMPLETE: implement the TODO and rerun')

<div dir="rtl">
<h2>فقط یک عامل را تغییر دهید</h2>
<p>فقط تعداد تکرار ID یک را تغییر دهید. جدول تازه با همان مقدارهای ثابت بسازید تا تفاوت Gradient فقط از تعداد استفاده بیاید. اینجا from_pretrained مقدار آغاز جدول را از Tensor می‌گیرد و freeze=False یعنی عددهای جدول قابل آموزش بمانند.</p>
</div>

In [ ]:
for sequence in [[1, 3, 1], [1, 3, 1, 1]]:
    embedding = nn.Embedding.from_pretrained(table.clone(), freeze=False)
    embedding(torch.tensor([sequence])).sum().backward()
    print(sequence, "row-1 gradient:", embedding.weight.grad[1].tolist())
model = TokenOnly(vocab_size=5, channels=3)
print("Actual v1 logits shape:", tuple(model(ids).shape))

<div dir="rtl">
<h2>خرابی را پیدا کنید</h2>
<p>میانگین‌گرفتن سهم‌های یک ID، قاعدهٔ جمع شاخه‌ها را عوض می‌کند. تابع row_counts(ids,V) تعداد وقوع هر ID را برگرداند؛ برای Loss جمع خروجی، این عدد Gradient هر مؤلفهٔ همان سطر است.</p>
</div>

In [ ]:
wrong = [int(i in ids.flatten().tolist()) for i in range(5)]
print("Presence is not usage count:", wrong)
assert wrong[1] == 1

<div dir="rtl">
<h2>اصلاح را خودتان بنویسید</h2>
<p>علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def row_counts(ids, V):
    # TODO: count every occurrence, not just presence
    return None

In [ ]:
def test_repair():
    result = row_counts(ids, 5)
    if result is None:
        return False
    assert result == [0, 2, 0, 1, 0]
    assert row_counts(torch.tensor([[0, 0], [2, 0]]), 3) == [3, 0, 1]
    return True

repair_complete = test_repair()
print('PASS' if repair_complete else 'INCOMPLETE: implement the TODO and rerun')

<div dir="rtl">
<h2>در Mini-GPT کجا به کار می‌آید؟</h2>
<p>TokenOnly از mini_gpt/stages/v1.py واقعاً اجرا شد: Embedding سپس Head. این مدل هنوز اطلاعات موقعیت‌های دیگر را ترکیب نمی‌کند و شباهت معنایی جدول تصادفی را ادعا نمی‌کنیم.</p>
</div>

<div dir="rtl">
<h2>با زبان خودتان توضیح دهید</h2>
<p>چه چیزی آموختنی است: ID، عمل انتخاب سطر، یا مقدارهای جدول؟</p>
</div>
<div dir="rtl"><p>پیش‌بینی و مشاهدهٔ من: …</p><p>علت خرابی و اصلاح من: …</p></div>

<div dir="rtl"><p><a target="_self" href="http://127.0.0.1:8000/part-04/chapter-03/25-embedding.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/25-embedding.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>